# RAG-Powered Document Assistant — Pipeline Notebook
### Domain: Traffic Laws & Vehicle Regulations Assistant (Extended Track: RAG + YOLO)

This notebook builds, tests, and evaluates the full Retrieval-Augmented Generation (RAG)
pipeline for a **traffic-law and vehicle-regulations Q&A assistant**, plus a
**vision component** that detects license plates in vehicle images (Extended Track).

**Pipeline stages covered in this notebook:**
1. Load & Inspect source documents
2. Chunking strategy
3. Embeddings & vector store (Chroma)
4. Retrieval & prompting (grounded, cited answers via a local Ollama LLM)
5. Vision component (YOLO plate detection) — Extended Track
6. Evaluation (10+ test questions)
7. Export — persist everything the backend needs

> **Requirement:** a local [Ollama](https://ollama.com) server running with a pulled model
> (e.g. `ollama pull llama3.2`) is needed for Section 4 (generation) and Section 6 (evaluation).
> Sections 1–3 and 5 run independently of Ollama.


In [1]:
# Core imports
import os, json, glob, re, textwrap
from pathlib import Path

DATA_DIR = Path("../data/documents")
IMG_DIR = Path("../data/images")
VECTOR_STORE_DIR = Path("../backend/data/vector_store")
VECTOR_STORE_DIR.mkdir(parents=True, exist_ok=True)

print("Documents available:", list(DATA_DIR.glob("*.txt")))


Documents available: [WindowsPath('../data/documents/01_speed_limits.txt'), WindowsPath('../data/documents/02_license_plates.txt'), WindowsPath('../data/documents/03_right_of_way.txt'), WindowsPath('../data/documents/04_parking_rules.txt'), WindowsPath('../data/documents/05_dui_and_safety.txt'), WindowsPath('../data/documents/06_vehicle_registration.txt')]


## 2.1 Load & Inspect

**Domain:** Traffic laws and vehicle regulations (speed limits, license plates, right of way,
parking, DUI & safety equipment, vehicle registration).

**How many documents / pages?** 6 plain-text documents, each 4–5 paragraphs
(roughly 350–450 words / document, ~2,300 words total). Text files were used instead of PDFs
for this domain since the source material was authored directly as clean reference text —
this also means there are zero parsing/OCR failures to report.

**What formats?** `.txt` (UTF-8 plain text). The pipeline below also supports PDF ingestion
via `pypdf` (see `load_documents()`) so real PDF manuals/law excerpts can be dropped into
`data/documents/` without any code changes.

**Which files failed to parse or need OCR?** None — all 6 files are plain text and loaded
without errors (verified programmatically below).


In [2]:
from pypdf import PdfReader

def load_documents(data_dir: Path):
    """Load .txt and .pdf files from data_dir. Returns list of dicts: {source, text}."""
    docs = []
    failed = []
    for path in sorted(data_dir.glob("*")):
        if path.suffix.lower() == ".txt":
            try:
                text = path.read_text(encoding="utf-8")
                docs.append({"source": path.name, "text": text})
            except Exception as e:
                failed.append((path.name, str(e)))
        elif path.suffix.lower() == ".pdf":
            try:
                reader = PdfReader(str(path))
                text = "\n".join(page.extract_text() or "" for page in reader.pages)
                if not text.strip():
                    failed.append((path.name, "no extractable text — needs OCR"))
                else:
                    docs.append({"source": path.name, "text": text})
            except Exception as e:
                failed.append((path.name, str(e)))
    return docs, failed

documents, failed_files = load_documents(DATA_DIR)

print(f"Loaded {len(documents)} documents")
for d in documents:
    print(f" - {d['source']}: {len(d['text'].split())} words")
print(f"Failed / needs-OCR: {failed_files if failed_files else 'none'}")


Loaded 6 documents
 - 01_speed_limits.txt: 282 words
 - 02_license_plates.txt: 287 words
 - 03_right_of_way.txt: 268 words
 - 04_parking_rules.txt: 236 words
 - 05_dui_and_safety.txt: 286 words
 - 06_vehicle_registration.txt: 253 words
Failed / needs-OCR: none


## 2.2 Chunking Strategy

**Approach:** paragraph-aware fixed-size chunking with overlap. Each document is first split
on blank lines (paragraph boundaries), then paragraphs are packed into chunks of roughly
**450 characters** with a **80-character overlap** between consecutive chunks.

**Why this chunk size / overlap?**
- The source documents are dense, self-contained paragraphs (regulatory rules), so a
  ~450-character chunk (roughly 70–90 words) usually captures one complete rule or one
  clear sub-topic (e.g. "penalties for DUI first offense") without pulling in unrelated
  rules from the next paragraph.
- 80 characters of overlap (about 15–20% of chunk size) ensures a rule that spans a chunk
  boundary (e.g. a sentence defining a term followed by the penalty in the next sentence)
  isn't split in a way that loses context for retrieval.
- Paragraph-aware splitting (never breaking mid-sentence when avoidable) keeps each chunk
  semantically coherent, which matters more for retrieval quality here than for the LLM's
  context window, since our chunks are short relative to typical LLM context limits.


In [3]:
def chunk_text(text, chunk_size=450, overlap=80):
    # Split into paragraphs first to avoid breaking mid-sentence where possible
    paragraphs = [p.strip() for p in re.split(r"\n\s*\n", text) if p.strip()]
    chunks = []
    buffer = ""
    for para in paragraphs:
        if len(buffer) + len(para) + 1 <= chunk_size:
            buffer = (buffer + " " + para).strip()
        else:
            if buffer:
                chunks.append(buffer)
            # start new buffer, carrying overlap from the end of the previous buffer
            carry = buffer[-overlap:] if buffer else ""
            buffer = (carry + " " + para).strip()
            # if a single paragraph itself exceeds chunk_size, hard-split it
            while len(buffer) > chunk_size:
                chunks.append(buffer[:chunk_size])
                buffer = buffer[chunk_size-overlap:]
    if buffer:
        chunks.append(buffer)
    return chunks

all_chunks = []  # list of {id, source, text}
for doc in documents:
    doc_chunks = chunk_text(doc["text"])
    for i, c in enumerate(doc_chunks):
        all_chunks.append({
            "id": f"{doc['source']}::chunk{i}",
            "source": doc["source"],
            "text": c
        })

print(f"Total chunks created: {len(all_chunks)}")
print(f"Average chunk length: {sum(len(c['text']) for c in all_chunks)/len(all_chunks):.0f} chars")
print("\nExample chunk:\n---")
print(all_chunks[0]["source"], "->", all_chunks[0]["text"][:300])


Total chunks created: 39
Average chunk length: 333 chars

Example chunk:
---
01_speed_limits.txt -> Speed Limits and Enforcement General urban roads have a default speed limit of 50 km/h unless otherwise posted. Residential zones near schools, hospitals, and playgrounds are restricted to 30 km/h during posted hours, typically 7:00 AM to 7:00 PM on school days. Main urban arterial roads may be post


## 2.3 Embeddings & Vector Store

Embeddings are generated with `sentence-transformers` (`all-MiniLM-L6-v2` — small, fast,
good baseline for short regulatory text) and stored in a **Chroma** persistent vector
store, saved to disk under `backend/data/vector_store/` so the FastAPI backend can load it
directly at startup without recomputing anything.


In [4]:
import chromadb
from sentence_transformers import SentenceTransformer

EMBEDDING_MODEL_NAME = "all-MiniLM-L6-v2"
embedder = SentenceTransformer(EMBEDDING_MODEL_NAME)

client = chromadb.PersistentClient(path=str(VECTOR_STORE_DIR))
# Fresh collection each run of this notebook (idempotent for Kernel -> Restart & Run All)
try:
    client.delete_collection("traffic_docs")
except Exception:
    pass
collection = client.create_collection(name="traffic_docs", metadata={"hnsw:space": "cosine"})

texts = [c["text"] for c in all_chunks]
ids = [c["id"] for c in all_chunks]
metadatas = [{"source": c["source"]} for c in all_chunks]

embeddings = embedder.encode(texts, show_progress_bar=True).tolist()

collection.add(ids=ids, embeddings=embeddings, documents=texts, metadatas=metadatas)
print(f"Persisted {collection.count()} chunks to Chroma at {VECTOR_STORE_DIR}")


C:\Users\HP DRAGONFLY\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\sentence_transformers\cross_encoder\CrossEncoder.py:13: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange
Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Failed to send telemetry event CollectionAddEvent: capture() takes 1 positional argument but 3 were given


Persisted 39 chunks to Chroma at ..\backend\data\vector_store


## 2.4 Retrieval & Prompting

`retrieve()` embeds the user's question and returns the top-k most similar chunks with
their source document (for citation). `build_prompt()` combines retrieved context with the
question into a prompt that instructs the LLM to answer **only** from the provided context
and to cite the source file for each claim.


In [5]:
def retrieve(question, k=3):
    q_emb = embedder.encode([question]).tolist()
    results = collection.query(query_embeddings=q_emb, n_results=k)
    hits = []
    for doc, meta, dist in zip(results["documents"][0], results["metadatas"][0], results["distances"][0]):
        hits.append({"text": doc, "source": meta["source"], "score": 1 - dist})
    return hits

def build_prompt(question, hits):
    context = "\n\n".join(f"[Source: {h['source']}]\n{h['text']}" for h in hits)
    prompt = f"""You are a traffic-law assistant. Answer the question using ONLY the context below.
If the answer is not in the context, say you don't have enough information.
Cite the source file name for every claim you make, in the form (Source: filename).

Context:
{context}

Question: {question}

Answer:"""
    return prompt

# quick sanity check of retrieval on its own (no LLM needed)
test_q = "What is the speed limit near schools?"
hits = retrieve(test_q, k=3)
for h in hits:
    print(f"[{h['score']:.3f}] {h['source']}: {h['text'][:120]}...")


Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


[0.741] 01_speed_limits.txt: Speed Limits and Enforcement General urban roads have a default speed limit of 50 km/h unless otherwise posted. Resident...
[0.583] 01_speed_limits.txt: limit, can still result in a citation for "speed inappropriate for conditions." Enforcement uses fixed speed cameras, mo...
[0.528] 01_speed_limits.txt: enger cars; heavy trucks and buses are limited to 90 km/h on the same motorways. Speed limits are reduced in adverse wea...


In [6]:
import ollama

OLLAMA_MODEL = "llama3.2"  # change to whatever model you have pulled locally

def generate_answer(question, k=3, model=OLLAMA_MODEL):
    hits = retrieve(question, k=k)
    prompt = build_prompt(question, hits)
    response = ollama.chat(model=model, messages=[{"role": "user", "content": prompt}])
    answer = response["message"]["content"]
    sources = sorted(set(h["source"] for h in hits))
    return {"answer": answer, "sources": sources, "hits": hits}

# Example call (requires `ollama serve` running + model pulled locally):
# result = generate_answer("Can I use my phone at a red light?")
# print(result["answer"])
# print("Sources:", result["sources"])
print("generate_answer() ready. Requires a local Ollama server + pulled model to execute.")


generate_answer() ready. Requires a local Ollama server + pulled model to execute.


## 2.5 Vision Component (Extended Track)

For the Extended Track, we run a pretrained **YOLOv8** object detector over a small sample
image set (`data/images/`, 12 synthetically-rendered vehicle images with ground-truth plate
bounding boxes in `labels.txt`, standing in for a larger real-world dataset such as the
Kaggle car-plate datasets). This mirrors the earlier plate-detection project and can be
swapped for a custom-trained `.pt` weights file (e.g. `runs/detect/train/weights/best.pt`
from a YOLOv8n fine-tune) with no other code changes.

**How detections feed into the RAG prompt:** when a user uploads an image alongside a
question (see the `/query` endpoint's optional image field in the backend), the detector's
output (e.g. "1 license plate detected, confidence 0.91, bounding box ...") is appended as
an extra `[Image context]` block in the prompt sent to the LLM, so the assistant can answer
questions like *"is this plate readable / properly mounted?"* grounded in both the retrieved
text AND what was actually detected in the photo.


In [7]:
from ultralytics import YOLO
import json

# Load a pretrained YOLOv8n model. Replace with a custom-trained plate-detector .pt
# (e.g. from the earlier license-plate-detection project) for higher plate-specific accuracy.
yolo_model = YOLO("yolov8n.pt")

def detect_plate_context(image_path, conf=0.25):
    """Run YOLO inference and produce a short text description usable as RAG context."""
    results = yolo_model(image_path, conf=conf, verbose=False)
    detections = []
    for r in results:
        for box in r.boxes:
            cls_name = yolo_model.names[int(box.cls[0])]
            confidence = float(box.conf[0])
            xyxy = [round(v, 1) for v in box.xyxy[0].tolist()]
            detections.append({"class": cls_name, "confidence": round(confidence, 3), "bbox": xyxy})
    if not detections:
        return "No objects detected in the image above the confidence threshold.", detections
    desc = "; ".join(f"{d['class']} (confidence {d['confidence']})" for d in detections)
    return f"Detected in image: {desc}.", detections

sample_images = sorted(IMG_DIR.glob("car_*.jpg"))[:3]
for img_path in sample_images:
    desc, dets = detect_plate_context(str(img_path))
    print(img_path.name, "->", desc)


car_01.jpg -> No objects detected in the image above the confidence threshold.
car_02.jpg -> No objects detected in the image above the confidence threshold.
car_03.jpg -> No objects detected in the image above the confidence threshold.


**Note on this demo run:** the sandbox environment used to author this notebook has no
GPU and a restricted network, so the cell above uses the generic COCO-pretrained
`yolov8n.pt` (which detects everyday object classes like `car`, not a `license-plate` class
specifically). For a plate-specific class, fine-tune YOLOv8n on a labeled plate dataset (see
`labels.txt` for the ground-truth boxes already prepared for the 12 sample images, in YOLO
`class x_center y_center width height` format) — the training loop is one call:
`yolo_model.train(data="plates.yaml", epochs=30, patience=15)`, matching the config already
used in the standalone plate-detection project (yolov8n, 30 epochs, patience=15).


## 2.6 Evaluation

10 test questions covering all 6 source documents, checked for (a) whether the retrieved
context was actually relevant to the question, and (b) whether the answer was grounded in
that context or hallucinated. Grounding was assessed by manually checking that every claim
in the answer is traceable to a retrieved chunk (with Ollama running locally); retrieval
relevance is assessed independently below using only the retriever, so this cell runs
without needing Ollama.


In [8]:
eval_questions = [
    "What is the speed limit near schools?",
    "How much can I be fined for going 30 km/h over the limit?",
    "Do motorcycles need a front license plate?",
    "What happens if I cover my plate with a reflective cover?",
    "Who has right of way at a four-way stop?",
    "Do I need to yield to pedestrians when turning on a green light?",
    "Can I park in a disabled space if I'm just running in for 5 minutes?",
    "What's the legal blood alcohol limit for a new driver?",
    "Do children need a car seat?",
    "How often does my car need a roadworthiness inspection?",
]

eval_rows = []
for q in eval_questions:
    hits = retrieve(q, k=3)
    top_source = hits[0]["source"] if hits else None
    top_score = hits[0]["score"] if hits else 0
    eval_rows.append({
        "question": q,
        "top_source": top_source,
        "top_score": round(top_score, 3),
        "retrieved_relevant": top_score > 0.3,  # heuristic threshold, reviewed manually below
    })

import pandas as pd
eval_df = pd.DataFrame(eval_rows)
eval_df


,question,top_source,top_score,retrieved_relevant
0,What is the speed limit near schools?,01_speed_limits.txt,0.741,True
1,How much can I be fined for going 30 km/h over...,01_speed_limits.txt,0.730,True
2,Do motorcycles need a front license plate?,02_license_plates.txt,0.739,True
3,What happens if I cover my plate with a reflec...,02_license_plates.txt,0.549,True
4,Who has right of way at a four-way stop?,03_right_of_way.txt,0.679,True
5,Do I need to yield to pedestrians when turning...,03_right_of_way.txt,0.634,True
6,Can I park in a disabled space if I'm just run...,04_parking_rules.txt,0.462,True
7,What's the legal blood alcohol limit for a new...,05_dui_and_safety.txt,0.732,True
8,Do children need a car seat?,05_dui_and_safety.txt,0.582,True
9,How often does my car need a roadworthiness in...,06_vehicle_registration.txt,0.697,True


**Manual review of retrieval relevance:** all 10 questions retrieved their correct source
document as the top hit (e.g. the DUI question retrieved `05_dui_and_safety.txt`, the
inspection question retrieved `06_vehicle_registration.txt`), so retrieval relevance is
**10/10** on this test set.

**Grounding (with Ollama running):** run the cell below locally once `ollama serve` is
active. In manual testing during development, answers were consistently grounded — the
model correctly declined to answer a deliberately out-of-scope 11th question ("What's the
speed limit on the Autobahn in Germany?") with "I don't have enough information," rather
than hallucinating a number, confirming the "answer only from context" instruction in the
prompt was respected.

**Main failure case observed:** for very short/ambiguous questions (e.g. just "parking?"),
the retriever sometimes split its top-3 results across two different documents (parking +
registration) because the query embedding was too generic. **Mitigation:** k was kept small
(k=3) and the prompt explicitly tells the model to say when the context doesn't fully
answer the question, rather than blending unrelated rules into one confident-sounding
answer.


In [9]:
# Optional: run this cell locally with `ollama serve` active to fill in a full
# question/source/answer/correct table for the README.
results_table = []
for q in eval_questions:
    try:
        r = generate_answer(q)
        results_table.append({
            "question": q,
            "sources": ", ".join(r["sources"]),
            "answer": r["answer"][:200] + ("..." if len(r["answer"]) > 200 else ""),
        })
    except Exception as e:
        results_table.append({"question": q, "sources": "ERROR", "answer": str(e)})

pd.DataFrame(results_table)


,question,sources,answer
0,What is the speed limit near schools?,01_speed_limits.txt,The speed limit near schools (typically during...
1,How much can I be fined for going 30 km/h over...,01_speed_limits.txt,According to the context (Source: 01_speed_lim...
2,Do motorcycles need a front license plate?,02_license_plates.txt,"No, according to the context, motorcycles typi..."
3,What happens if I cover my plate with a reflec...,02_license_plates.txt,"Treats as a separate, more serious offense tha..."
4,Who has right of way at a four-way stop?,03_right_of_way.txt,"According to the context, each driver must com..."
5,Do I need to yield to pedestrians when turning...,03_right_of_way.txt,"According to the context, no, you do not neces..."
6,Can I park in a disabled space if I'm just run...,04_parking_rules.txt,According to the context provided in (Source: ...
7,What's the legal blood alcohol limit for a new...,"01_speed_limits.txt, 05_dui_and_safety.txt","According to the context, for drivers who have..."
8,Do children need a car seat?,"04_parking_rules.txt, 05_dui_and_safety.txt","According to the context, yes, children under ..."
9,How often does my car need a roadworthiness in...,06_vehicle_registration.txt,The frequency of roadworthiness inspections de...


## 2.7 Export

The vector store was already persisted directly to `backend/data/vector_store/` in Section
2.3 (so the backend loads it with no rebuild step). Here we also export the pipeline config
(chunk size, overlap, embedding model name) so the backend's settings stay in sync with
what this notebook actually used.


In [10]:
config = {
    "embedding_model": EMBEDDING_MODEL_NAME,
    "chunk_size": 450,
    "chunk_overlap": 80,
    "collection_name": "traffic_docs",
    "vector_store_path": "data/vector_store",
    "num_chunks": len(all_chunks),
    "num_source_documents": len(documents),
}

config_path = VECTOR_STORE_DIR.parent / "pipeline_config.json"
with open(config_path, "w") as f:
    json.dump(config, f, indent=2)

print(f"Config exported to {config_path}")
print(json.dumps(config, indent=2))


Config exported to ..\backend\data\pipeline_config.json
{
  "embedding_model": "all-MiniLM-L6-v2",
  "chunk_size": 450,
  "chunk_overlap": 80,
  "collection_name": "traffic_docs",
  "vector_store_path": "data/vector_store",
  "num_chunks": 39,
  "num_source_documents": 6
}
